In [7]:
import pandas as pd
import numpy as np
from libpysal.weights import full2W
from esda.getisord import G_Local
from pyprojroot import here

In [8]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [9]:
crime = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', encoding='latin-1', index_col=0, dtype=str)

In [10]:
crime["CVEGEO"] = crime["CVEGEO"].astype(str).str.zfill(5)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [11]:
common_ids = sorted(set(adj.index) & set(crime["CVEGEO"]))
adj = adj.loc[common_ids, common_ids]

In [12]:
w = full2W(adj.values.astype(int), ids=common_ids)
w.transform = "r" 

print(f"True islands: {w.islands}") 

True islands: []


In [14]:
ALPHA = 0.05
Z_CRITICAL = 1.96 

all_years_results = []
years = sorted(crime["year"].unique())

for year in years:
    yr_data = crime[crime["year"] == year].set_index("CVEGEO").reindex(common_ids)

    if yr_data["crime_count"].isna().any():
        missing = yr_data[yr_data["crime_count"].isna()].index.tolist()
        print(f"Year {year}: missing crime_count for {len(missing)} municipalities — skipping year")
        continue

    y = yr_data["crime_count"].values.astype(float)

    g_local = G_Local(y, w, transform='R', permutations=999, seed=42)

    sig = g_local.p_sim < ALPHA

    conditions = [
        (sig) & (g_local.Zs > 0),  
        (sig) & (g_local.Zs < 0),  
        (~sig)                     
    ]

    choices = ['HS', 'CS', 'ns']
    cluster = np.select(conditions, choices, default='ns')
    
    year_df = pd.DataFrame({
        "CVEGEO": common_ids,
        "year": year,
        "crime_count": y,
        "observed_g": g_local.Gs,
        "z_score": g_local.Zs,        
        "cluster": cluster,          
        "p_sim": g_local.p_sim,
        "significant": sig,
    })

    all_years_results.append(year_df)

    n_sig = sig.sum()
    print(f"Year {year}: {n_sig}/{len(common_ids)} municipalities significant at alpha={ALPHA}")

local_g_results = pd.concat(all_years_results, ignore_index=True)

local_g_results = local_g_results.merge(
    crime[["CVEGEO", "state", "municipality"]].drop_duplicates(),
    on="CVEGEO", how="left"
)

print(local_g_results.head())


Year 2015: 773/2478 municipalities significant at alpha=0.05
Year 2016: 693/2478 municipalities significant at alpha=0.05
Year 2017: 680/2478 municipalities significant at alpha=0.05
Year 2018: 597/2478 municipalities significant at alpha=0.05
Year 2019: 587/2478 municipalities significant at alpha=0.05
Year 2020: 630/2478 municipalities significant at alpha=0.05
Year 2021: 621/2478 municipalities significant at alpha=0.05
Year 2022: 597/2478 municipalities significant at alpha=0.05
Year 2023: 575/2478 municipalities significant at alpha=0.05
Year 2024: 622/2478 municipalities significant at alpha=0.05
Year 2025: 634/2478 municipalities significant at alpha=0.05
  CVEGEO  year  crime_count  observed_g   z_score cluster  p_sim  significant  \
0  01001  2015        863.0    0.000542  0.057733      ns  0.223        False   
1  01002  2015          8.0    0.002313  0.804268      HS  0.040         True   
2  01003  2015         17.0    0.002569  0.911988      HS  0.034         True   
3  01

In [15]:
local_g_results.to_csv(RESULTS_DATA_DIR/"raw_esda_results/yearly_glocal_results.csv", encoding='latin-1', index=False)

In [16]:
summary = (
    local_g_results
    .groupby(["year", "cluster"])
    .size()
    .unstack(fill_value=0)
)
print(summary)
summary.to_csv(RESULTS_DATA_DIR/"raw_esda_results/yearly_localg_cluster_summary.csv")

cluster    CS   HS    ns
year                    
2015     1057  162  1814
2016      905  162  1966
2017      892  163  1978
2018      697  148  2188
2019      678  161  2194
2020      718  182  2133
2021      730  182  2121
2022      699  175  2159
2023      661  170  2202
2024      720  166  2147
2025      731  187  2115
